# ATM Cash Forecasting - Demo Notebook

This notebook demonstrates the ATM cash forecasting pipeline including:
- Data generation and loading
- Feature engineering
- Model training
- Evaluation with stockout costs
- Visualization

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

from src.utils.helpers import load_config
from src.utils.data_generator import generate_synthetic_data
from src.data.ingestion import DataIngestion, DataPreprocessor
from src.features.engineering import FeatureEngineering
from src.models.lightgbm_model import LightGBMForecaster
from src.evaluation.backtesting import BacktestEngine, ModelEvaluator

# Configure plots
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Load Configuration

In [ ]:
config = load_config('../config.yaml')
print("Configuration loaded successfully!")

## 2. Generate Sample Data

In [ ]:
# Generate synthetic data for 50 ATMs
df = generate_synthetic_data(
    n_atms=50,
    start_date='2022-01-01',
    end_date='2023-12-31',
    output_path='../data/raw/demo_data.csv'
)

print(f"Generated {len(df)} records for {df['atm_id'].nunique()} ATMs")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Select one ATM for detailed analysis
sample_atm = df['atm_id'].unique()[0]
df_atm = df[df['atm_id'] == sample_atm].copy()

# Plot time series
plt.figure(figsize=(15, 5))
plt.plot(df_atm['date'], df_atm['cash_demand'])
plt.title(f'Cash Demand Time Series - {sample_atm}')
plt.xlabel('Date')
plt.ylabel('Cash Demand (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Statistics
print(f"\nStatistics for {sample_atm}:")
print(df_atm['cash_demand'].describe())

## 4. Data Preprocessing

In [ ]:
preprocessor = DataPreprocessor(imputation_method='linear')
df_clean = preprocessor.preprocess(df, handle_outliers=True)

print("Data preprocessing completed!")
print(f"Missing values after imputation: {df_clean['cash_demand'].isnull().sum()}")

## 5. Feature Engineering

In [ ]:
feature_config = {
    'lag_periods': [1, 7, 14, 30],
    'rolling_windows': [7, 14, 30],
    'use_holidays': True,
    'use_covariance': False,
    'country': 'IN',
    'covariance_window': 30
}

feature_eng = FeatureEngineering(feature_config)
df_features = feature_eng.engineer_features(df_clean, target_col='cash_demand', atm_id_col='atm_id')

print(f"Created {len(df_features.columns)} columns")
print(f"\nFeature columns: {[col for col in df_features.columns if col not in ['date', 'atm_id', 'cash_demand']]}")

## 6. Train-Test Split

In [ ]:
# Filter to one ATM
df_atm_features = df_features[df_features['atm_id'] == sample_atm].sort_values('date').reset_index(drop=True)

# Split
backtest_engine = BacktestEngine(config['backtesting'])
train_df, test_df = backtest_engine.train_test_split(df_atm_features)

print(f"Train size: {len(train_df)} records")
print(f"Test size: {len(test_df)} records")

## 7. Model Training

In [ ]:
# Initialize model
lgbm_model = LightGBMForecaster(config['models']['lightgbm'])

# Identify feature columns
exclude_cols = ['cash_demand', 'date', 'atm_id']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

print(f"Training with {len(feature_cols)} features...")

# Train
lgbm_model.fit(train_df, target_col='cash_demand', feature_cols=feature_cols)

print("Model training completed!")

## 8. Predictions and Evaluation

In [ ]:
# Make predictions
y_test = test_df['cash_demand'].values
y_pred = lgbm_model.predict(test_df)
y_train = train_df['cash_demand'].values

# Evaluate
evaluator = ModelEvaluator(config['costs'])
metrics = evaluator.evaluate_model(y_test, y_pred, y_train)

# Display metrics
print("\nModel Performance Metrics:")
print("=" * 50)
for metric, value in metrics.items():
    print(f"{metric:20s}: {value:,.2f}")

## 9. Visualize Predictions

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Time series plot
axes[0].plot(test_df['date'], y_test, label='Actual', marker='o')
axes[0].plot(test_df['date'], y_pred, label='Predicted', marker='x')
axes[0].set_title('Actual vs Predicted Cash Demand')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cash Demand (INR)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot
axes[1].scatter(y_test, y_pred, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_title('Prediction Scatter Plot')
axes[1].set_xlabel('Actual Cash Demand (INR)')
axes[1].set_ylabel('Predicted Cash Demand (INR)')

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# Get feature importance
feature_importance = lgbm_model.get_feature_importance()

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Display table
print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

## 11. Error Analysis

In [ ]:
# Calculate errors
errors = y_test - y_pred
percentage_errors = (errors / y_test) * 100

# Plot error distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram of errors
axes[0].hist(errors, bins=30, edgecolor='black')
axes[0].set_title('Distribution of Prediction Errors')
axes[0].set_xlabel('Error (INR)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(x=0, color='r', linestyle='--', label='Zero Error')
axes[0].legend()

# Histogram of percentage errors
axes[1].hist(percentage_errors, bins=30, edgecolor='black')
axes[1].set_title('Distribution of Percentage Errors')
axes[1].set_xlabel('Error (%)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=0, color='r', linestyle='--', label='Zero Error')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean Absolute Error: ₹{np.abs(errors).mean():,.0f}")
print(f"Mean Percentage Error: {percentage_errors.mean():.2f}%")

## 12. Conclusion

This notebook demonstrated the complete ATM cash forecasting pipeline:
1. Data generation and loading
2. Preprocessing and imputation
3. Feature engineering with lags, rolling windows, and holidays
4. Model training with LightGBM
5. Evaluation with business-relevant cost metrics
6. Feature importance analysis

Next steps:
- Compare with other models (Prophet, N-BEATS)
- Scale to multiple ATMs
- Implement real-time forecasting
- Deploy with Streamlit UI